# Demo 6: An evaluation report that gates on regression, and a trace walkthrough that finds the cause of one failure

**Outcome:** (1) an evaluation report that **gates on regression**, and (2) a **trace walkthrough** that locates the cause of one failure.

**Idea in one line:** you changed something in your agent. Before shipping, run a fixed test set against the old and new versions, **block the release if anything got worse**, and when something fails, read the **trace** (the step-by-step record of the run) to see *which step* went wrong.

This follows the same shape as the Google Cloud agent-evaluation material (ADK `EvalSet` with eval cases, a tool-trajectory metric, a response metric, multi-turn conversations, and the Gen AI evaluation service's rubric/AutoRater idea). The service's agent-evaluation features are Pre-GA, so this notebook builds the framework by hand on a plain EvalSet-style structure; it runs locally in seconds.

| Cloud concept | In this notebook |
|---|---|
| ADK `EvalSet` / eval cases (conversation + expected tool use + reference) | `EVAL_SET`, section 3 |
| Tool-trajectory metric | `score_case`: did the agent call the expected tools, in order? |
| Final-response metric | `score_case`: does the reply contain the expected facts? |
| Multi-turn evaluation | case `c4`: "Where is A200?" then "ok cancel it" (needs memory of the order id) |
| Traces / observability | `spans` recorded for every run, section 2 |
| Regression gate in CI | `gate()`, section 5 |
| AutoRater / LLM-as-judge | talking points (swap `score_case` for a judge prompt) |

**The agent:** a tiny order-support agent with two tools, `lookup_order` and `cancel_order`. The model only **decides** which tool to call and **words the answer**; the tools are plain code.
**The story:** version `v1` is the shipped agent. Version `v2` is a "harmless refactor" of the lookup tool. The eval set catches that it broke something.

## 0. Setup: mock model by default, any real model by changing three lines
Same switch as the earlier demos. The mock is a deterministic stand-in so the notebook runs anywhere; a real model plugs into `ask_model`.

In [1]:
import json, os, re

BACKEND  = "mock"                      # "mock" (default) or "real"
BASE_URL = "http://localhost:11434/v1" # Ollama. Also: Groq, OpenRouter, LM Studio (any OpenAI-compatible server)
MODEL    = "llama3.1"                  # whatever your server serves
API_KEY  = "ollama"                    # Ollama ignores it; Groq/OpenRouter need a real key
OUT_DIR  = "eval_out"

def ask_model(prompt):
    """The ONLY place the model is called. Returns a string."""
    if BACKEND == "mock":
        return mock_model(prompt)
    from openai import OpenAI          # pip install openai
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
    r = client.chat.completions.create(model=MODEL, temperature=0,
                                       messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content.strip()

ID = re.compile(r"\b[Aa-zA-Z]\s?-?\d{3}\b")

def mock_model(prompt):
    """Deterministic stand-in for an LLM. Handles the two kinds of prompt the agent sends."""
    users = [l[6:] for l in prompt.splitlines() if l.startswith("USER: ")]
    if "TASK=DECIDE" in prompt:
        latest = users[-1].lower()
        ids = [m.group(0) for u in reversed(users) for m in [ID.search(u)] if m]   # newest id mentioned in the conversation
        if not ids:
            return "none"
        if "cancel" in latest:
            return f"cancel_order {ids[0]}"
        if any(w in latest for w in ("where", "status", "track")):
            return f"lookup_order {ids[0]}"
        return "none"
    # TASK=ANSWER: word a reply from the tool result
    result = prompt.split("TOOL_RESULT: ", 1)[1].splitlines()[0]
    if result == "none":
        return "I can only help with orders. How can I help?"
    r = json.loads(result)
    if r.get("error") == "not_found":     return "Sorry, I couldn't find that order."
    if r.get("error") == "not_cancellable": return f"Order {r['order_id']} can't be cancelled because it is {r['status']}."
    if r.get("cancelled"):                return f"Order {r['order_id']} has been cancelled."
    return f"Order {r['order_id']} is {r['status']}."

## 1. The agent and its two versions
`TOOLSETS` holds the tool code of each version. **`v2` differs from `v1` in one line:** the "refactor" dropped the `.upper()` that made order ids case-insensitive.

In [2]:
ORDERS = {"A100": "shipped", "A200": "processing", "A300": "delivered"}

# ---- v1: the shipped version -------------------------------------------------
def lookup_v1(order_id):
    oid = order_id.strip().upper().replace(" ", "").replace("-", "")
    return {"order_id": oid, "status": ORDERS[oid]} if oid in ORDERS else {"error": "not_found"}

def cancel_v1(order_id):
    oid = order_id.strip().upper().replace(" ", "").replace("-", "")
    if oid not in ORDERS:
        return {"error": "not_found"}
    if ORDERS[oid] != "processing":
        return {"error": "not_cancellable", "order_id": oid, "status": ORDERS[oid]}
    return {"order_id": oid, "cancelled": True}

# ---- v2: the "harmless refactor": lookup no longer normalises the id ----------
def lookup_v2(order_id):
    oid = order_id.strip()                                   # <- .upper() was removed
    return {"order_id": oid, "status": ORDERS[oid]} if oid in ORDERS else {"error": "not_found"}

TOOLSETS = {
    "v1": {"lookup_order": lookup_v1, "cancel_order": cancel_v1},
    "v2": {"lookup_order": lookup_v2, "cancel_order": cancel_v1},
}
print("versions:", list(TOOLSETS))

versions: ['v1', 'v2']


## 2. Running the agent and recording a trace
Every run records **spans**, one per step: `user_input`, `model_decision`, `tool_call`, `tool_result`, `model_answer`. The trace is what lets you point at the step that went wrong.

In [3]:
DECIDE = ("TASK=DECIDE\nYou are an order-support agent. Tools: lookup_order(order_id), cancel_order(order_id).\n"
          "{conversation}\n"
          "Reply with ONE line only: 'lookup_order <id>', 'cancel_order <id>' or 'none'.")
ANSWER = ("TASK=ANSWER\nYou are an order-support agent.\n{conversation}\nTOOL_RESULT: {result}\n"
          "Reply to the customer in one short sentence, using only the tool result.")

def parse_decision(text):
    """Tolerant parser: find a tool name, then the id after it. Anything else means no tool."""
    m = re.search(r"(lookup_order|cancel_order)\W+(.*)", text.replace("*", ""), re.I)
    if not m:
        return None, None
    idm = re.search(r"[a-z]\s?-?\d{3}", m.group(2), re.I)          # keep the id exactly as the model wrote it
    return (m.group(1).lower(), idm.group(0).replace(" ", "") if idm else None)

def run_case(turns, version):
    tools, spans, history, replies, calls = TOOLSETS[version], [], [], [], []
    for user_text in turns:
        history.append(f"USER: {user_text}")
        spans.append({"kind": "user_input", "data": user_text})
        conv = "\n".join(history)
        decision = ask_model(DECIDE.format(conversation=conv))
        name, arg = parse_decision(decision)
        spans.append({"kind": "model_decision", "data": f"{name} {arg}" if name and arg else "none"})
        if name and arg:
            spans.append({"kind": "tool_call", "data": f"{name}({arg})"})
            result = tools[name](arg)
            calls.append((name, arg))
        else:
            result = "none"
        spans.append({"kind": "tool_result", "data": json.dumps(result) if result != "none" else "none"})
        reply = ask_model(ANSWER.format(conversation=conv, result=spans[-1]["data"]))
        spans.append({"kind": "model_answer", "data": reply})
        history.append(f"ASSISTANT: {reply}")
        replies.append(reply)
    return {"replies": replies, "tool_calls": calls, "spans": spans}

demo = run_case(["Where is order A100?"], "v1")
print("One run, as a trace:")
for s in demo["spans"]:
    print(f"  {s['kind']:<15} {s['data']}")

One run, as a trace:
  user_input      Where is order A100?
  model_decision  lookup_order A100
  tool_call       lookup_order(A100)
  tool_result     {"order_id": "A100", "status": "shipped"}
  model_answer    Order A100 is shipped.


## 3. The eval set and the two metrics
An eval set is a list of cases: the user turns, the **tools we expect** (the trajectory) and **facts the final reply must contain**. Two metrics per case, like ADK's tool-trajectory and response checks. A case passes only if both pass.

In [4]:
EVAL_SET = [
  {"id": "c1", "turns": ["Where is order A100?"],                  "expect_tools": ["lookup_order"],                 "reply_has": ["shipped"]},
  {"id": "c2", "turns": ["Status of A300 please"],                 "expect_tools": ["lookup_order"],                 "reply_has": ["delivered"]},
  {"id": "c3", "turns": ["where is order a200"],                   "expect_tools": ["lookup_order"],                 "reply_has": ["processing"]},     # lowercase id
  {"id": "c4", "turns": ["Where is A200?", "ok cancel it"],        "expect_tools": ["lookup_order", "cancel_order"], "reply_has": ["cancelled"]},      # multi-turn
  {"id": "c5", "turns": ["track a100"],                            "expect_tools": ["lookup_order"],                 "reply_has": ["shipped"]},        # lowercase id
  {"id": "c6", "turns": ["Where is B999?"],                        "expect_tools": ["lookup_order"],                 "reply_has": ["couldn't find"]},
  {"id": "c7", "turns": ["What is the weather today?"],            "expect_tools": [],                              "reply_has": ["help"]},
]

def score_case(case, run):
    """Returns the two metrics and whether the case passed."""
    trajectory_ok = [n for n, _ in run["tool_calls"]] == case["expect_tools"]
    final = run["replies"][-1].lower()
    response_ok = all(fact.lower() in final for fact in case["reply_has"])
    return {"trajectory": trajectory_ok, "response": response_ok, "passed": trajectory_ok and response_ok}

def evaluate(version):
    """Run the whole eval set against one version of the agent."""
    results = {}
    for case in EVAL_SET:
        run = run_case(case["turns"], version)
        results[case["id"]] = {**score_case(case, run), "run": run}
    return results

def pass_rate(results): return sum(r["passed"] for r in results.values()) / len(results)

print(f"{len(EVAL_SET)} eval cases, ids:", [c["id"] for c in EVAL_SET])

7 eval cases, ids: ['c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7']


## 4. Run both versions on the same eval set

In [5]:
baseline  = evaluate("v1")      # what is in production
candidate = evaluate("v2")      # the change we want to ship

print(f"{'case':<5}{'v1 (baseline)':<16}{'v2 (candidate)':<16}")
for cid in baseline:
    f = lambda r: "pass" if r["passed"] else "FAIL"
    print(f"{cid:<5}{f(baseline[cid]):<16}{f(candidate[cid]):<16}")
print(f"\npass rate: v1 {pass_rate(baseline):.0%}   v2 {pass_rate(candidate):.0%}")

case v1 (baseline)   v2 (candidate)  
c1   pass            pass            
c2   pass            pass            
c3   pass            FAIL            
c4   pass            pass            
c5   pass            FAIL            
c6   pass            pass            
c7   pass            pass            

pass rate: v1 100%   v2 71%


## 5. The regression gate
Two rules, both must hold to ship:
1. **The pass rate must not drop** below the baseline.
2. **No case that passed before may fail now.** This matters: if the change fixes one case and breaks another, the pass rate looks unchanged, but a user-visible behaviour was lost.

In [6]:
def gate(baseline_results, candidate_results, max_drop=0.0):
    newly_failing = [cid for cid in baseline_results
                     if baseline_results[cid]["passed"] and not candidate_results[cid]["passed"]]
    drop = pass_rate(baseline_results) - pass_rate(candidate_results)
    reasons = []
    if drop > max_drop + 1e-9:
        reasons.append(f"pass rate fell by {drop:.0%} (allowed {max_drop:.0%})")
    if newly_failing:
        reasons.append(f"cases that passed before now fail: {newly_failing}")
    return {"decision": "BLOCK" if reasons else "SHIP", "reasons": reasons, "newly_failing": newly_failing}

def write_report(version_a, res_a, version_b, res_b, verdict, path_prefix="eval_report"):
    os.makedirs(OUT_DIR, exist_ok=True)
    lines = [f"# Evaluation report: {version_b} vs baseline {version_a}", "",
             f"**Gate decision: {verdict['decision']}**", ""]
    lines += [f"- {r}" for r in verdict["reasons"]] or ["- no regressions found"]
    lines += ["", f"Pass rate: {version_a} {pass_rate(res_a):.0%}, {version_b} {pass_rate(res_b):.0%}", "",
              f"| case | {version_a} trajectory | {version_a} response | {version_b} trajectory | {version_b} response | status |", "|---|---|---|---|---|---|"]
    mark = lambda ok: "ok" if ok else "FAIL"
    for cid in res_a:
        a, b = res_a[cid], res_b[cid]
        status = "REGRESSION" if cid in verdict["newly_failing"] else ("ok" if b["passed"] else "still failing")
        lines.append(f"| {cid} | {mark(a['trajectory'])} | {mark(a['response'])} | {mark(b['trajectory'])} | {mark(b['response'])} | {status} |")
    md = "\n".join(lines)
    open(f"{OUT_DIR}/{path_prefix}.md", "w").write(md)
    json.dump({"decision": verdict["decision"], "reasons": verdict["reasons"],
               "pass_rate": {version_a: pass_rate(res_a), version_b: pass_rate(res_b)},
               "regressions": verdict["newly_failing"]}, open(f"{OUT_DIR}/{path_prefix}.json", "w"), indent=2)
    return md

verdict = gate(baseline, candidate)
print(write_report("v1", baseline, "v2", candidate, verdict))
print("\nCI exit code would be:", 1 if verdict["decision"] == "BLOCK" else 0)

# Evaluation report: v2 vs baseline v1

**Gate decision: BLOCK**

- pass rate fell by 29% (allowed 0%)
- cases that passed before now fail: ['c3', 'c5']

Pass rate: v1 100%, v2 71%

| case | v1 trajectory | v1 response | v2 trajectory | v2 response | status |
|---|---|---|---|---|---|
| c1 | ok | ok | ok | ok | ok |
| c2 | ok | ok | ok | ok | ok |
| c3 | ok | ok | ok | FAIL | REGRESSION |
| c4 | ok | ok | ok | ok | ok |
| c5 | ok | ok | ok | FAIL | REGRESSION |
| c6 | ok | ok | ok | ok | ok |
| c7 | ok | ok | ok | ok | ok |

CI exit code would be: 1


## 6. Trace walkthrough: where exactly did it go wrong?
The gate says *that* something broke. The trace says *where*. Compare the good run (v1) and the bad run (v2) of the same case, step by step, and find the **first step that differs**. Everything before it is identical, so the cause is at that step.

In [7]:
CAUSE_BY_STEP = {
    "user_input":     "the input itself differs, so the test setup changed",
    "model_decision": "the model chose a different action: look at the prompt or the model",
    "tool_call":      "the model passed different arguments to the tool",
    "tool_result":    "same call but a different result: the TOOL (its code or data) changed",
    "model_answer":   "same tool result but a different answer: look at the answer prompt or the model",
}

def first_divergence(good_spans, bad_spans):
    """Index of the first step whose kind or data differs; None if the traces are identical."""
    for i, (g, b) in enumerate(zip(good_spans, bad_spans)):
        if (g["kind"], g["data"]) != (b["kind"], b["data"]):
            return i
    return None if len(good_spans) == len(bad_spans) else min(len(good_spans), len(bad_spans))

def walkthrough(case_id):
    good, bad = baseline[case_id]["run"]["spans"], candidate[case_id]["run"]["spans"]
    i = first_divergence(good, bad)
    print(f"Case {case_id}: {EVAL_SET[[c['id'] for c in EVAL_SET].index(case_id)]['turns']}\n")
    print(f"  {'step':<4}{'kind':<16}{'v1 (good)':<48}{'v2 (bad)'}")
    for k, (g, b) in enumerate(zip(good, bad)):
        flag = "  <-- first difference" if k == i else ""
        print(f"  {k:<4}{g['kind']:<16}{g['data'][:44]:<48}{b['data'][:44]}{flag}")
    if i is not None:
        kind = good[i]["kind"]
        print(f"\nDiagnosis: steps 0-{i-1} are identical in both runs; the first difference is step {i} ({kind}).")
        print(f"           {CAUSE_BY_STEP[kind]}.")
    return i

failing = verdict["newly_failing"]
focus = failing[0] if failing else None
step = walkthrough(focus) if focus else print("No regression this run, nothing to walk through.")

Case c3: ['where is order a200']

  stepkind            v1 (good)                                       v2 (bad)
  0   user_input      where is order a200                             where is order a200
  1   model_decision  lookup_order a200                               lookup_order a200
  2   tool_call       lookup_order(a200)                              lookup_order(a200)
  3   tool_result     {"order_id": "A200", "status": "processing"}    {"error": "not_found"}  <-- first difference
  4   model_answer    Order A200 is processing.                       Sorry, I couldn't find that order.

Diagnosis: steps 0-2 are identical in both runs; the first difference is step 3 (tool_result).
           same call but a different result: the TOOL (its code or data) changed.


### Reading the walkthrough
The model made the **same decision** and called the **same tool with the same argument** (`lookup_order(a200)`). Only the tool's **result** differs: v1 normalised the id and found the order, v2 did not and said `not_found`. The model then correctly reported what it was told. So the cause is the tool change, not the model or the prompt, which is why the fix goes in `lookup_v2`, not in the prompt.

## 7. Fix it and re-run the gate
Restore the normalisation, evaluate again, and the same gate now lets it ship.

In [8]:
def lookup_v3(order_id):
    return lookup_v1(order_id)                         # fix: ids are normalised again
TOOLSETS["v3"] = {"lookup_order": lookup_v3, "cancel_order": cancel_v1}

fixed = evaluate("v3")
verdict_fixed = gate(baseline, fixed)
print(write_report("v1", baseline, "v3", fixed, verdict_fixed, path_prefix="eval_report_after_fix"))

# Evaluation report: v3 vs baseline v1

**Gate decision: SHIP**

- no regressions found

Pass rate: v1 100%, v3 100%

| case | v1 trajectory | v1 response | v3 trajectory | v3 response | status |
|---|---|---|---|---|---|
| c1 | ok | ok | ok | ok | ok |
| c2 | ok | ok | ok | ok | ok |
| c3 | ok | ok | ok | ok | ok |
| c4 | ok | ok | ok | ok | ok |
| c5 | ok | ok | ok | ok | ok |
| c6 | ok | ok | ok | ok | ok |
| c7 | ok | ok | ok | ok | ok |


## 8. Tests

In [9]:
def check(name, condition):
    print(("PASS  " if condition else "FAIL  ") + name)
    assert condition, name

# 1. The eval set itself is well formed
ids = [c["id"] for c in EVAL_SET]
check("1. eval set is well formed: unique ids, every case has turns, expected tools and expected facts",
      len(ids) == len(set(ids)) and all(c["turns"] and "expect_tools" in c and c["reply_has"] for c in EVAL_SET))

# 2. The baseline is healthy (otherwise "regression" has no meaning)
print(f"   baseline pass rate: {pass_rate(baseline):.0%}")
check("2. baseline v1 passes at least 80% of the eval set (with the mock model: all of it)", pass_rate(baseline) >= 0.8)

# 3. The metrics themselves work: a wrong tool fails the trajectory metric, a missing fact fails the response metric
case = {"expect_tools": ["lookup_order"], "reply_has": ["shipped"]}
good = {"tool_calls": [("lookup_order", "A100")], "replies": ["Order A100 is shipped."]}
wrong_tool = {"tool_calls": [("cancel_order", "A100")], "replies": ["Order A100 is shipped."]}
wrong_text = {"tool_calls": [("lookup_order", "A100")], "replies": ["Sorry, I couldn't find that order."]}
check("3. metrics: correct run passes; wrong tool fails trajectory; missing fact fails response",
      score_case(case, good)["passed"] and not score_case(case, wrong_tool)["trajectory"] and not score_case(case, wrong_text)["response"])

# 4. The gate on the real runs: its decision agrees with the data
check("4. gate decision on v1 vs v2 is consistent: BLOCK exactly when a case regressed or the pass rate fell",
      (verdict["decision"] == "BLOCK") == bool(verdict["newly_failing"] or pass_rate(candidate) < pass_rate(baseline)))
print(f"   gate on v2: {verdict['decision']}  {verdict['reasons']}")

# 5. The gate catches a swap: one case fixed, another broken, pass rate unchanged
mk = lambda *oks: {f"c{i}": {"passed": ok} for i, ok in enumerate(oks)}
swap = gate(mk(True, False, True), mk(False, True, True))
check("5. gate BLOCKS when the pass rate is equal but a previously passing case now fails", swap["decision"] == "BLOCK")

# 6. The gate lets a harmless change through, and a pure improvement
check("6. gate SHIPS when results are identical, or strictly better",
      gate(mk(True, True), mk(True, True))["decision"] == "SHIP" and gate(mk(True, False), mk(True, True))["decision"] == "SHIP")

# 7. Trace comparison finds the first differing step exactly
a = [{"kind": "user_input", "data": "x"}, {"kind": "model_decision", "data": "lookup_order a1"}, {"kind": "tool_result", "data": "ok"}]
b = [{"kind": "user_input", "data": "x"}, {"kind": "model_decision", "data": "lookup_order a1"}, {"kind": "tool_result", "data": "bad"}]
check("7. first_divergence returns the index of the first differing step (2), and None for identical traces",
      first_divergence(a, b) == 2 and first_divergence(a, a) is None)

# 8. The walkthrough on the real regression localises it, and everything before it really is identical
if failing:
    good_s, bad_s = baseline[focus]["run"]["spans"], candidate[focus]["run"]["spans"]
    i = first_divergence(good_s, bad_s)
    print(f"   first difference in case {focus}: step {i} ({good_s[i]['kind']})")
    check("8. regression localised: all steps before the first difference are identical in v1 and v2",
          i is not None and all((g["kind"], g["data"]) == (b_["kind"], b_["data"]) for g, b_ in zip(good_s[:i], bad_s[:i])))
    if BACKEND == "mock":
        check("8b. (mock) the cause is the tool_result step: same decision, same call, different result", good_s[i]["kind"] == "tool_result")
else:
    print("SKIP  8. no regression in this run (a real model may have normalised the id itself)")

# 9. Fixing the cause fixes the gate
check("9. after the fix (v3) the gate SHIPS and no case regressed", verdict_fixed["decision"] == "SHIP" and not verdict_fixed["newly_failing"])

# 10. The report files exist and say what the gate decided
rep = open(f"{OUT_DIR}/eval_report.md").read()
js = json.load(open(f"{OUT_DIR}/eval_report.json"))
check("10. report written: markdown and JSON agree with the gate decision and name the regressed cases",
      f"Gate decision: {verdict['decision']}" in rep and js["decision"] == verdict["decision"] and js["regressions"] == verdict["newly_failing"])

# 11. The parser copes with messy model output
check("11. decision parser handles messy replies like 'I will call **lookup_order** with A300.' and rejects chatter",
      parse_decision("I will call **lookup_order** with A300.") == ("lookup_order", "A300") and parse_decision("Sure, happy to help!") == (None, None))

print("\nOutcome met: a regression gate with a written report, and a trace walkthrough that finds the cause of one failure.")

PASS  1. eval set is well formed: unique ids, every case has turns, expected tools and expected facts
   baseline pass rate: 100%
PASS  2. baseline v1 passes at least 80% of the eval set (with the mock model: all of it)
PASS  3. metrics: correct run passes; wrong tool fails trajectory; missing fact fails response
PASS  4. gate decision on v1 vs v2 is consistent: BLOCK exactly when a case regressed or the pass rate fell
   gate on v2: BLOCK  ['pass rate fell by 29% (allowed 0%)', "cases that passed before now fail: ['c3', 'c5']"]
PASS  5. gate BLOCKS when the pass rate is equal but a previously passing case now fails
PASS  6. gate SHIPS when results are identical, or strictly better
PASS  7. first_divergence returns the index of the first differing step (2), and None for identical traces
   first difference in case c3: step 3 (tool_result)
PASS  8. regression localised: all steps before the first difference are identical in v1 and v2
PASS  8b. (mock) the cause is the tool_result step: s